# Phase 2 / V2 — Results dashboard

این notebook فقط نتایج مدل‌هایی را مقایسه می‌کند که واقعاً اجرا شده‌اند. مقایسهٔ clip-level و full-MP4 عمداً جدا نگه داشته می‌شود، چون سؤال اصلی محصول، full-MP4 است.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_ROOT = Path(r'P:\\NexarCollisionData')
MODEL_DIR = DATA_ROOT / 'models_v2'
INFERENCE_DIR = DATA_ROOT / 'inference_v2'
REPORT_DIR = DATA_ROOT / 'reports_v2'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CLIP_CSV_PATH = REPORT_DIR / 'phase2_v2_clip_metrics.csv'
FULL_CSV_PATH = REPORT_DIR / 'phase2_v2_full_mp4_metrics.csv'
STATUS_CSV_PATH = REPORT_DIR / 'phase2_v2_model_status.csv'
SUMMARY_PATH = REPORT_DIR / 'phase2_v2_results_summary.json'
REPORT_PATH = REPORT_DIR / 'phase2_v2_results_analysis.md'
DASHBOARD_PATH = REPORT_DIR / 'phase2_v2_results_dashboard.png'
HISTORY_PATH = REPORT_DIR / 'phase2_v2_training_pr_auc_history.png'

for required_path in (
    MODEL_DIR / 'resnet18_mean_pooling_frozen_metrics.json',
    MODEL_DIR / 'resnet18_meanmax_pooling_frozen_metrics.json',
    MODEL_DIR / 'resnet18_temporal_attention_frozen_metrics.json',
    MODEL_DIR / 'resnet18_gru_frozen_metrics.json',
    MODEL_DIR / 'resnet18_gru_attention_frozen_metrics.json',
    MODEL_DIR / 'resnet18_meanmax_finetuned_layer4_metrics.json',
    MODEL_DIR / 'resnet18_meanmax_pooling_frozen_multipos_clip_metrics.json',
    INFERENCE_DIR / 'a2_vs_a2_multipos_full_video_comparison.csv',
    INFERENCE_DIR / 'a2_multipos_calibration_report.json',
    INFERENCE_DIR / 'a2_multipos_final_error_summary.json',
):
    assert required_path.exists(), f'Missing required result: {required_path}'

In [2]:
clip_specs = [
    ('A1', 'ResNet18 + mean pooling', 'resnet18_mean_pooling_frozen_metrics.json', 'resnet18_mean_pooling_frozen_training_history.csv'),
    ('A2', 'ResNet18 + mean-max pooling', 'resnet18_meanmax_pooling_frozen_metrics.json', 'resnet18_meanmax_pooling_frozen_training_history.csv'),
    ('A3', 'ResNet18 + temporal attention', 'resnet18_temporal_attention_frozen_metrics.json', None),
    ('A4', 'ResNet18 + GRU', 'resnet18_gru_frozen_metrics.json', 'resnet18_gru_frozen_training_history.csv'),
    ('A5', 'ResNet18 + GRU + attention', 'resnet18_gru_attention_frozen_metrics.json', 'resnet18_gru_attention_frozen_training_history.csv'),
    ('A2-FT', 'A2 with layer4 fine-tuning', 'resnet18_meanmax_finetuned_layer4_metrics.json', 'resnet18_meanmax_finetuned_layer4_training_history.csv'),
    ('A2-MP', 'A2 with three event positions', 'resnet18_meanmax_pooling_frozen_multipos_clip_metrics.json', 'resnet18_meanmax_pooling_frozen_multipos_training_history.csv'),
]

clip_records = []
history_specs = []
for order, (short_name, display_name, metrics_filename, history_filename) in enumerate(clip_specs):
    payload = json.loads((MODEL_DIR / metrics_filename).read_text(encoding='utf-8'))
    threshold_key = next(key for key in payload if key.startswith('selected_threshold_by_'))
    metrics = payload['metrics_at_selected_threshold']
    clip_records.append({
        'order': order, 'model_id': short_name, 'model': display_name,
        'evaluation_scope': 'clip-level fixed validation',
        'threshold': float(payload[threshold_key]), 'best_epoch': int(payload['best_epoch']),
        **{key: float(metrics[key]) for key in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc')},
        'confusion_matrix': metrics['confusion_matrix'],
    })
    if history_filename and (MODEL_DIR / history_filename).exists():
        history_specs.append((short_name, MODEL_DIR / history_filename))

clip_metrics = pd.DataFrame(clip_records).sort_values('order').reset_index(drop=True)
clip_metrics.to_csv(CLIP_CSV_PATH, index=False)
display(clip_metrics[['model_id', 'model', 'f1', 'recall', 'precision', 'pr_auc', 'threshold', 'best_epoch']])

,model_id,model,f1,recall,precision,pr_auc,threshold,best_epoch
0,A1,ResNet18 + mean pooling,0.762712,0.750000,0.775862,0.727511,0.47,23
1,A2,ResNet18 + mean-max pooling,0.787402,0.833333,0.746269,0.742437,0.44,25
2,A3,ResNet18 + temporal attention,0.773723,0.883333,0.688312,0.716780,0.39,7
3,A4,ResNet18 + GRU,0.753623,0.866667,0.666667,0.761337,0.17,9
4,A5,ResNet18 + GRU + attention,0.761194,0.850000,0.689189,0.783847,0.25,18
5,A2-FT,A2 with layer4 fine-tuning,0.731707,1.000000,0.576923,0.747465,0.20,3
6,A2-MP,A2 with three event positions,0.761194,0.850000,0.689189,0.730450,0.30,12


In [3]:
full_metrics = pd.read_csv(INFERENCE_DIR / 'a2_vs_a2_multipos_full_video_comparison.csv').copy()
full_metrics['model_id'] = ['A2', 'A2-MP']
full_metrics['evaluation_scope'] = 'full-MP4 sliding-window validation'
full_metrics.to_csv(FULL_CSV_PATH, index=False)

calibration = json.loads((INFERENCE_DIR / 'a2_multipos_calibration_report.json').read_text(encoding='utf-8'))
error_summary = json.loads((INFERENCE_DIR / 'a2_multipos_final_error_summary.json').read_text(encoding='utf-8'))

status_rows = [
    {'model_id': 'A0', 'model': 'ResNet18 V1, 8 scattered frames', 'status': 'reference_only', 'reason': 'V1 reference; not part of the fixed V2 clip/full-MP4 comparison.'},
    *[{'model_id': row.model_id, 'model': row.model, 'status': 'executed', 'reason': 'Metrics recorded in this report.'} for row in clip_metrics.itertuples()],
    {'model_id': 'A6', 'model': 'ResNet18 + BiLSTM + additive attention + FFN', 'status': 'not_executed', 'reason': 'Planned next architecture.'},
    {'model_id': 'B1', 'model': 'R3D-18 pretrained', 'status': 'not_executed', 'reason': 'Requires CUDA GPU for practical training.'},
    {'model_id': 'B2', 'model': 'R(2+1)D-18 pretrained', 'status': 'not_executed', 'reason': 'Requires CUDA GPU for practical training.'},
    {'model_id': 'C1', 'model': 'VideoMAE', 'status': 'pipeline_ready_not_executed', 'reason': 'Preflight pipeline is prepared; transformers and CUDA are not available locally.'},
    {'model_id': 'D1', 'model': 'Best model + frame difference', 'status': 'not_executed', 'reason': 'Planned motion ablation.'},
    {'model_id': 'D2', 'model': 'Best model + metadata fusion', 'status': 'not_executed', 'reason': 'Planned metadata ablation.'},
    {'model_id': 'E1', 'model': 'Ensemble', 'status': 'not_executed', 'reason': 'Requires complementary completed base models.'},
]
model_status = pd.DataFrame(status_rows)
model_status.to_csv(STATUS_CSV_PATH, index=False)

print('Full-MP4 comparison:')
display(full_metrics[['model_id', 'model', 'f1', 'recall', 'precision', 'pr_auc', 'roc_auc', 'aggregation', 'threshold']])
print('Model status:')
display(model_status)

Full-MP4 comparison:


,model_id,model,f1,recall,precision,pr_auc,roc_auc,aggregation,threshold
0,A2,A2 original training,0.737589,0.866667,0.641975,0.70807,0.723889,max,0.57
1,A2-MP,A2-MP three event positions,0.737589,0.866667,0.641975,0.72267,0.740278,top3_mean,0.40


Model status:


,model_id,model,status,reason
0,A0,"ResNet18 V1, 8 scattered frames",reference_only,V1 reference; not part of the fixed V2 clip/fu...
1,A1,ResNet18 + mean pooling,executed,Metrics recorded in this report.
2,A2,ResNet18 + mean-max pooling,executed,Metrics recorded in this report.
3,A3,ResNet18 + temporal attention,executed,Metrics recorded in this report.
4,A4,ResNet18 + GRU,executed,Metrics recorded in this report.
5,A5,ResNet18 + GRU + attention,executed,Metrics recorded in this report.
6,A2-FT,A2 with layer4 fine-tuning,executed,Metrics recorded in this report.
7,A2-MP,A2 with three event positions,executed,Metrics recorded in this report.
8,A6,ResNet18 + BiLSTM + additive attention + FFN,not_executed,Planned next architecture.
9,B1,R3D-18 pretrained,not_executed,Requires CUDA GPU for practical training.


In [4]:
def grouped_metric_bars(axis, table: pd.DataFrame, metric_columns: list[str], title: str) -> None:
    positions = np.arange(len(table))
    width = 0.18
    offsets = np.linspace(-1.5 * width, 1.5 * width, len(metric_columns))
    for offset, metric in zip(offsets, metric_columns):
        bars = axis.bar(positions + offset, table[metric], width, label=metric.replace('_', ' ').upper())
        axis.bar_label(bars, fmt='%.2f', fontsize=7, padding=2, rotation=90)
    axis.set_xticks(positions, table['model_id'])
    axis.set_ylim(0.45, 1.05)
    axis.set_ylabel('Score')
    axis.set_title(title)
    axis.grid(axis='y', alpha=0.25)
    axis.legend(fontsize=8, ncol=2)

figure, axes = plt.subplots(2, 2, figsize=(16, 11))
grouped_metric_bars(axes[0, 0], clip_metrics, ['f1', 'recall', 'precision', 'pr_auc'], 'Clip-level validation: executed V2 models')
grouped_metric_bars(axes[0, 1], full_metrics, ['f1', 'recall', 'precision', 'pr_auc'], 'Full-MP4 validation: A2 vs A2-MP')

error_labels = ['True positive', 'True negative', 'False positive', 'False negative']
error_values = [error_summary['true_positive'], error_summary['true_negative'], error_summary['false_positive'], error_summary['false_negative']]
error_colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']
bars = axes[1, 0].bar(error_labels, error_values, color=error_colors)
axes[1, 0].bar_label(bars, padding=3)
axes[1, 0].set_ylim(0, 60)
axes[1, 0].set_ylabel('Validation videos')
axes[1, 0].set_title('Final A2-MP full-MP4 decision breakdown')
axes[1, 0].tick_params(axis='x', rotation=18)
axes[1, 0].grid(axis='y', alpha=0.25)

axes[1, 1].axis('off')
text = '\n'.join([
    'Final selected configuration',
    'A2-MP: frozen ResNet18 + mean-max pooling',
    'Input: 16 frames / 5-second window',
    f"Aggregation: {full_metrics.loc[full_metrics['model_id'].eq('A2-MP'), 'aggregation'].iloc[0]}",
    f"Raw decision threshold: {error_summary['raw_threshold']:.2f}",
    f"F1 / Recall: {full_metrics.loc[full_metrics['model_id'].eq('A2-MP'), 'f1'].iloc[0]:.3f} / {full_metrics.loc[full_metrics['model_id'].eq('A2-MP'), 'recall'].iloc[0]:.3f}",
    f"PR-AUC: {full_metrics.loc[full_metrics['model_id'].eq('A2-MP'), 'pr_auc'].iloc[0]:.3f}",
    f"Uncertain videos: {error_summary['uncertain_videos']} / 120",
    f"CPU inference (40 s MP4): about 87 s",
])
axes[1, 1].text(0.04, 0.94, text, va='top', ha='left', fontsize=12, linespacing=1.7,
                bbox={'boxstyle': 'round,pad=0.8', 'facecolor': '#f3f6fb', 'edgecolor': '#5875a4'})
figure.suptitle('Problem 1 — Phase 2 / V2 experimental dashboard', fontsize=16, y=0.98)
figure.tight_layout(rect=(0, 0, 1, 0.96))
figure.savefig(DASHBOARD_PATH, dpi=180)
plt.close(figure)
print(f'Dashboard: {DASHBOARD_PATH}')

Dashboard: P:\NexarCollisionData\reports_v2\phase2_v2_results_dashboard.png


In [5]:
figure, axis = plt.subplots(figsize=(11, 5))
for model_id, history_path in history_specs:
    history = pd.read_csv(history_path)
    if 'validation_pr_auc' in history.columns:
        axis.plot(history['epoch'], history['validation_pr_auc'], marker='o', markersize=3, label=model_id)
axis.set(xlabel='Epoch', ylabel='Validation PR-AUC', title='Training history of models with saved histories', ylim=(0.60, 0.90))
axis.grid(alpha=0.3)
axis.legend(title='Model', ncol=4)
figure.tight_layout()
figure.savefig(HISTORY_PATH, dpi=180)
plt.close(figure)
print(f'Training history chart: {HISTORY_PATH}')

Training history chart: P:\NexarCollisionData\reports_v2\phase2_v2_training_pr_auc_history.png


In [6]:
a2_clip = clip_metrics.loc[clip_metrics['model_id'].eq('A2')].iloc[0]
a2mp_clip = clip_metrics.loc[clip_metrics['model_id'].eq('A2-MP')].iloc[0]
a2_full = full_metrics.loc[full_metrics['model_id'].eq('A2')].iloc[0]
a2mp_full = full_metrics.loc[full_metrics['model_id'].eq('A2-MP')].iloc[0]

summary = {
    'dataset_scope': '600 selected videos; 480 train and 120 fixed validation; balanced in both splits',
    'clip_models_executed': clip_metrics['model_id'].tolist(),
    'clip_best_f1_model': str(clip_metrics.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]['model_id']),
    'clip_best_pr_auc_model': str(clip_metrics.sort_values('pr_auc', ascending=False).iloc[0]['model_id']),
    'full_mp4_selected_model': 'A2-MP',
    'full_mp4_primary_metrics': {key: float(a2mp_full[key]) for key in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc')},
    'full_mp4_config': {'aggregation': str(a2mp_full['aggregation']), 'raw_threshold': float(a2mp_full['threshold'])},
    'calibration': {
        'temperature': calibration['final_temperature_fitted_on_all_validation_videos'],
        'raw_ece': calibration['ece_raw'],
        'cross_fitted_calibrated_ece': calibration['ece_cross_fitted_calibrated'],
        'uncertain_videos': error_summary['uncertain_videos'],
    },
    'error_counts': {key: error_summary[key] for key in ('true_positive', 'true_negative', 'false_positive', 'false_negative')},
    'warning': 'All metrics are development metrics on the same fixed validation split. They are not an independent final-test estimate.',
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

lines = [
        '# نتایج و تحلیل فاز دوم (V2) — مسئلهٔ ۱', '',
        '## دامنهٔ آزمایش', '',
        '- ۶۰۰ ویدئوی انتخاب‌شده و متوازن: ۳۰۰ دارای تصادف و ۳۰۰ بدون تصادف.',
        '- تقسیم ثابت در سطح ویدئو: ۴۸۰ train و ۱۲۰ validation، هر کلاس ۶۰ نمونه در validation.',
        '- ورودی V2: ۱۶ فریم RGB با اندازهٔ ۲۲۴×۳۲۰ از یک پنجرهٔ پنج‌ثانیه‌ای.',
        '- مدل‌ها در سطح sequence/video برچسب‌گذاری شدند؛ فریم‌ها به‌صورت مستقل برچسب نداشتند.', '',
        '## مدل‌های اجراشده — clip-level validation', '',
        '| مدل | F1 | Recall | Precision | PR-AUC | Threshold |',
        '|---|---:|---:|---:|---:|---:|',
    ]
for row in clip_metrics.itertuples():
    lines.append(f'| {row.model_id}: {row.model} | {row.f1:.3f} | {row.recall:.3f} | {row.precision:.3f} | {row.pr_auc:.3f} | {row.threshold:.2f} |')
lines.extend([
        '', 'A2 بالاترین F1 سطح clip را دارد. A3 و A4 recall بالاتری نشان دادند، اما با افت precision و F1 همراه بودند. A5 بالاترین PR-AUC clip را دارد، ولی معیار اصلی F1 آن از A2 کمتر است. fine-tuning لایهٔ چهارم نیز با وجود recall برابر ۱٫۰۰، false positive بسیار بیشتری ساخت و انتخاب نشد.', '',
        '## ارزیابی اصلی full-MP4', '',
        '| مدل | Aggregation | Threshold | F1 | Recall | Precision | ROC-AUC | PR-AUC |',
        '|---|---|---:|---:|---:|---:|---:|---:|',
    ])
for row in full_metrics.itertuples():
    lines.append(f'| {row.model_id}: {row.model} | {row.aggregation} | {row.threshold:.2f} | {row.f1:.3f} | {row.recall:.3f} | {row.precision:.3f} | {row.roc_auc:.3f} | {row.pr_auc:.3f} |')
lines.extend([
        '',
        'A2 و A2-MP در F1، Recall، Precision و confusion matrix برابر بودند. A2-MP با ROC-AUC و PR-AUC بالاتر، فقط در معیار ثانویهٔ رتبه‌بندی بهتر است؛ بنابراین مدل موقت منتخب A2-MP است، ولی این یک بهبود بزرگ یا قطعی در تشخیص گسسته نیست.', '',
        '## تنظیم probability و خطا', '',
        f"- دمای calibration: {calibration['final_temperature_fitted_on_all_validation_videos']:.3f}.",
        f"- ECE خام: {calibration['ece_raw']:.3f} و ECE calibrated به‌صورت cross-fitted: {calibration['ece_cross_fitted_calibrated']:.3f}.",
        f"- تصمیم رسمی همچنان probability خام با threshold {error_summary['raw_threshold']:.2f} است؛ threshold calibrated معادل {error_summary['calibrated_threshold_equivalent_to_raw_decision']:.3f} همان برچسب‌ها را برمی‌گرداند.",
        f"- full-MP4 final: TP={error_summary['true_positive']}، TN={error_summary['true_negative']}، FP={error_summary['false_positive']}، FN={error_summary['false_negative']} و {error_summary['uncertain_videos']} ویدئو نامطمئن هستند.",
        '- false positiveها عمدتاً شامل ترافیک فشرده، خودروهای نزدیک، چراغ ترمز، تقاطع، شب/باران و glare بودند.',
        '- false negativeها با glare شدید، شب، highway و رخدادهای کوچک‌تر همراه بودند. مکان پنجرهٔ با بیشترین احتمال فقط یک تشخیص کمکی است، نه خروجی رسمی مکان‌یابی.', '',
        '## وضعیت مدل‌های roadmap', '',
        '| شناسه | وضعیت |', '|---|---|',
    ])
for row in model_status.itertuples():
    lines.append(f'| {row.model_id} | {row.status}: {row.reason} |')
lines.extend([
        '', '## محدودیت‌های گزارش', '',
        '- همهٔ اعداد، development metrics روی همان validation ثابت ۱۲۰تایی هستند؛ چون مدل، threshold و aggregation با آن انتخاب شده‌اند، این‌ها برآورد مستقل عملکرد نهایی نیستند.',
        '- برای ادعای عمومی‌تر باید test مستقل یا cross-validation انجام شود.',
        '- سرعت inference روی CPU فعلی حدود ۸۷ ثانیه برای یک MP4 چهل‌ثانیه‌ای اندازه‌گیری شد؛ این برای دمو مناسب است، اما deployment بلادرنگ نیست.', '',
        '## فایل‌های خروجی', '',
        f'- Dashboard: {DASHBOARD_PATH}',
        f'- نمودار history: {HISTORY_PATH}',
        f'- جدول clip: {CLIP_CSV_PATH}',
        f'- جدول full-MP4: {FULL_CSV_PATH}',
        f'- وضعیت roadmap: {STATUS_CSV_PATH}',
    ])
REPORT_PATH.write_text('\n'.join(lines) + '\n', encoding='utf-8')

print(f'Summary JSON: {SUMMARY_PATH}')
print(f'Markdown report: {REPORT_PATH}')
display(clip_metrics[['model_id', 'f1', 'recall', 'precision', 'pr_auc']])
display(full_metrics[['model_id', 'f1', 'recall', 'precision', 'pr_auc']])

Summary JSON: P:\NexarCollisionData\reports_v2\phase2_v2_results_summary.json
Markdown report: P:\NexarCollisionData\reports_v2\phase2_v2_results_analysis.md


,model_id,f1,recall,precision,pr_auc
0,A1,0.762712,0.750000,0.775862,0.727511
1,A2,0.787402,0.833333,0.746269,0.742437
2,A3,0.773723,0.883333,0.688312,0.716780
3,A4,0.753623,0.866667,0.666667,0.761337
4,A5,0.761194,0.850000,0.689189,0.783847
5,A2-FT,0.731707,1.000000,0.576923,0.747465
6,A2-MP,0.761194,0.850000,0.689189,0.730450


,model_id,f1,recall,precision,pr_auc
0,A2,0.737589,0.866667,0.641975,0.70807
1,A2-MP,0.737589,0.866667,0.641975,0.72267
